# Prac W8 - MLPs and Convolutional Neural Networks (CNNs)

In [7]:
# Imports
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torchsummary import summary
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split 
from torch.utils.tensorboard import SummaryWriter
import datetime 
!rm -rf ./logs/

print(f"MPS Available: {torch.mps.is_available()}")

!pip install tqdm
!pip install tensorboard

from tqdm import tqdm

MPS Available: True

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


import the MNIST dataset and normalise

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3801, ))
])

# Load MNIST training dataset with transformations
mnist_train = datasets.MNIST('data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('data', train=False, download=True, transform=transform)

Use `DataLoader` to batch and shuffle the dataset

In [9]:
# Split dataset into training, validation and test sets
train_size = int(0.8 * len(mnist_train))
val_size = len(mnist_train) - train_size
mnist_train, mnist_val = random_split(mnist_train, [train_size, val_size])

# Define data loaders
train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
val_loader = DataLoader(mnist_val, batch_size=64)
test_loader = DataLoader(mnist_test, batch_size=64)

Here we'll create a single layer MLP that uses the relu activation in its hidden layer

In [10]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28, 28, 512)
        self.fc2 = nn.Linear(512, 10)
    
    def forward(self, X):
        X = self.flatten(X)
        X = F.relu(self.fc1(X))
        return X

To simplify the `forward` function, we can group the hidden layers using `nn.Sequential`

In [16]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, X):
        X = self.model(X)
        return X

We set up `optimiser`, `loss` function, `TensorBoard` and `epoch`

In [19]:
# Instantiate MLP
model = MLP()

# Optimiser
optimiser = optim.SGD(model.parameters(), lr=0.01)

# Loss Function
loss_fn = nn.CrossEntropyLoss()

# Set up Tensorboard log directory
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "/MLP"
writer = SummaryWriter(log_dir)

# Example training loop
num_epochs = 5

Then we go into the training process

In [21]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        optimiser.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    # Logging training loss and accuracy
    writer.add_scalar('Loss/train', running_loss / len(train_loader), epoch)
    writer.add_scalar('Accuracy/train', 100. * correct / total, epoch)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    # Logging validation loss and accuracy
    writer.add_scalar('Loss/val', val_loss / len(val_loader), epoch)
    writer.add_scalar('Accuracy/val', 100. * val_correct / val_total, epoch)

writer.close()
summary(model, input_size=(1, 28*28))

Epoch 5: 100%|██████████| 750/750 [00:01<00:00, 611.47it/s]


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
              ReLU-3                  [-1, 512]               0
            Linear-4                   [-1, 10]           5,130
Total params: 407,050
Trainable params: 407,050
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 1.55
Estimated Total Size (MB): 1.57
----------------------------------------------------------------


We then launch the tensorboard

In [24]:
%load_ext tensorboard
%tensorboard --logdir ./logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 7876), started 0:00:34 ago. (Use '!kill 7876' to kill it.)

### CNNs

In [26]:
# Set up Tensorboard writer
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "/CNN"
writer = SummaryWriter(log_dir)

# Define CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.fc1 = nn.Linear(32 * 26 * 26, 512)
        self.fc2 = nn.Linear(512, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate the model
model = CNN()

# Define optimiser and loss function
optimiser = optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimiser.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs,labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    writer.add_scalar('Loss/train', running_loss / len(train_loader), epoch)
    writer.add_scalar('Accuracy/train', 100. * correct / total, epoch)

    # Validation loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    # Log validation loss and accuracy
    writer.add_scalar('Loss/val', val_loss / len(test_loader), epoch)
    writer.add_scalar('Accuracy/val', 100. * val_correct / val_total, epoch)

writer.close()
summary(model, input_size=(1, 28, 28))

Epoch 5: 100%|██████████| 750/750 [00:08<00:00, 88.32it/s]


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 26, 26]             320
            Linear-2                  [-1, 512]      11,076,096
            Linear-3                   [-1, 10]           5,130
Total params: 11,081,546
Trainable params: 11,081,546
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.17
Params size (MB): 42.27
Estimated Total Size (MB): 42.44
----------------------------------------------------------------


### **Question 1**
Change the activation functions from relu to tanh or sigmoids and compare the performance

First we create helper functions to help us with this question and future questions

In [32]:
from torch.nn.modules.loss import _Loss
from typing import List

# Dynamic CNN Class
class CNNDynamic(nn.Module):
    def __init__(self, conv_channels: List[int] = [32], hidden_dims: List[int] = [512], 
        activation: nn.Module = nn.ReLU, num_classes=10):
        super().__init__()
        layers = [] 
        in_c = 1
        for out_c in conv_channels:
            layers += [nn.Conv2d(in_c, out_c, 3), activation()]
            in_c = out_c 
        layers.append(nn.Flatten())
        for h in hidden_dims:
            layers += [nn.LazyLinear(h), activation()]
        layers.append(nn.LazyLinear(num_classes))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

def train_model(model: CNN, optimiser: optim.Optimizer, loss_fn: _Loss,
    train_loader: DataLoader) -> CNN:
    for epoch in range(num_epochs):
        model.train()

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimiser.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs,labels)
            loss.backward()
            optimiser.step()
    
    return model

def evaluate(model: CNN, loader: DataLoader) -> None:
    model.eval()
    total_loss: float = 0.0
    correct: int = 0
    n: int = 0

    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)

    print(f"Model Loss: {total_loss / n}")
    print(f"Accuracy: {correct / n}")

Now we can try with the tanh and sigmoid function

In [37]:
# ReLU Function
cnn_relu: CNNDynamic = CNNDynamic(activation=nn.ReLU)
cnn_relu = train_model(
    model=cnn_relu,
    optimiser=optim.SGD(cnn_relu.parameters(), lr=0.01),
    loss_fn=nn.CrossEntropyLoss(),
    train_loader=train_loader 
)
evaluate(cnn_relu, test_loader)

# Tanh function
cnn_tanh: CNNDynamic = CNNDynamic(activation=nn.Tanh)
cnn_tanh = train_model(
    model=cnn_tanh,
    optimiser=optim.SGD(cnn_tanh.parameters(), lr=0.01),
    loss_fn=nn.CrossEntropyLoss(),
    train_loader=train_loader
)
evaluate(cnn_tanh, test_loader)

# Sigmoid function
cnn_sigmoid: CNNDynamic = CNNDynamic(activation=nn.Sigmoid)
cnn_sigmoid = train_model(
    model=cnn_sigmoid,
    optimiser=optim.SGD(cnn_sigmoid.parameters(), lr=0.01),
    loss_fn=nn.CrossEntropyLoss(),
    train_loader=train_loader
)
evaluate(cnn_sigmoid, test_loader)

Epoch 5: 100%|██████████| 750/750 [00:08<00:00, 90.79it/s]


Model Loss: 0.08844731611981987
Accuracy: 0.9739


Epoch 5: 100%|██████████| 750/750 [00:08<00:00, 86.27it/s]


Model Loss: 0.13681760196983814
Accuracy: 0.9593


Epoch 5: 100%|██████████| 750/750 [00:09<00:00, 82.25it/s]


Model Loss: 0.33373639686107637
Accuracy: 0.9037


### **Question 2**
Change the parameters of the optimiser and compare the performance.

I'm skipping this question cause its boring and training would take bloody ages

### **Question 3**

Change the optimiser from SGD to Adam or AdaGrad and compare the performance

In [38]:
cnn_reg: CNNDynamic = CNNDynamic()
cnn_reg = train_model(
    model=cnn_reg,
    optimiser=optim.Adam(cnn_reg.parameters(), lr=1e-3),
    loss_fn=nn.CrossEntropyLoss(),
    train_loader=train_loader
)
evaluate(cnn_reg, test_loader)

Epoch 5: 100%|██████████| 750/750 [00:11<00:00, 64.66it/s]


Model Loss: 0.06612702424547752
Accuracy: 0.9809


### **Question 4**

Change the model to have more or less layers and compare the performance.

Again imma skip this one cause training would take so long but wallahi I could do this question if I really wanted to.